In [1]:
%run code/database.py

数据库已准备好：C:\Users\34974\大三上-作业\task3\data\hotel.db


In [2]:
connection = get_connection()

tables = connection.execute(
    "SELECT name FROM sqlite_master "
    "WHERE type = 'table' "
    "ORDER BY name"
).fetchall()

print([table["name"] for table in tables])
connection.close()

['bookings', 'daily_inventory', 'daily_prices', 'hotels', 'operation_logs', 'room_types', 'rooms', 'sqlite_sequence', 'stop_sales', 'users']


In [7]:
%run code/auth_service.py

In [8]:
from auth_service import login, register_customer

customer = login("desti_demo", "DestiDemo123")

if customer is None:
    customer = register_customer(
        "desti_demo",
        "13800000001",
        "DestiDemo123",
    )

print(customer)

User(id=1, username='desti_demo', phone='13800000001', role='customer', is_active=True)


In [9]:
%run code/booking_service.py

In [10]:
from booking_service import create_booking

jiangjing_room = next(
    room for room in results
    if room["room_type_name"] == "江景大床房"
)

booking = create_booking(
    customer.id,
    jiangjing_room["room_type_id"],
    check_in.isoformat(),
    check_out.isoformat(),
    2,
    "邱雨翔",
    "13800000001",
)

print(booking)

Booking(id=1, order_number='DT261126184348F18B', customer_id=1, hotel_id=1, room_type_id=1, check_in='2026-11-27', check_out='2026-11-29', guest_count=2, contact_name='邱雨翔', contact_phone='13800000001', total_amount=1426.0, status='pending', cancel_reason=None)


In [11]:
updated_rooms = search_room_types(
    "上海",
    check_in.isoformat(),
    check_out.isoformat(),
    2,
)

for room in updated_rooms:
    if room["room_type_name"] == "江景大床房":
        print("当前可订房间数：", room["available_rooms"])

当前可订房间数： 5


In [12]:
%run code/booking_service.py

In [13]:
my_bookings = list_customer_bookings(customer.id)

for order in my_bookings:
    print(order)

{'id': 1, 'order_number': 'DT261126184348F18B', 'hotel_name': '外滩澜庭酒店', 'room_type_name': '江景大床房', 'check_in': '2026-11-27', 'check_out': '2026-11-29', 'guest_count': 2, 'total_amount': 1426.0, 'status': 'pending', 'cancel_reason': None}


In [14]:
print(
    cancel_booking(
        customer.id,
        booking.order_number,
        "用于测试取消功能",
    )
)

订单 DT261126184348F18B 已取消，房间库存已释放。


In [1]:
%run code/main.py

In [2]:
from pathlib import Path
import sqlite3

db_path = Path.cwd() / "data" / "hotel.db"
print("数据库位置：", db_path)

connection = sqlite3.connect(db_path)

tables = connection.execute(
    "SELECT name FROM sqlite_master "
    "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' "
    "ORDER BY name"
).fetchall()

for (table_name,) in tables:
    columns = connection.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    column_text = ", ".join(
        f"{column[1]} ({column[2]})" for column in columns
    )
    print(f"{table_name}: {column_text}")

connection.close()

数据库位置： C:\Users\34974\Documents\GitHub\hotel-reservation-system\task3\data\hotel.db
bookings: id (INTEGER), order_number (TEXT), customer_id (INTEGER), hotel_id (INTEGER), room_type_id (INTEGER), check_in (TEXT), check_out (TEXT), guest_count (INTEGER), contact_name (TEXT), contact_phone (TEXT), total_amount (REAL), status (TEXT), cancel_reason (TEXT), created_at (TEXT)
daily_inventory: id (INTEGER), room_type_id (INTEGER), stay_date (TEXT), total_rooms (INTEGER), booked_rooms (INTEGER)
daily_prices: id (INTEGER), room_type_id (INTEGER), stay_date (TEXT), price (REAL)
hotels: id (INTEGER), hotel_code (TEXT), name (TEXT), city (TEXT), address (TEXT), contact_phone (TEXT), is_active (INTEGER)
operation_logs: id (INTEGER), operator_id (INTEGER), booking_id (INTEGER), action (TEXT), details (TEXT), created_at (TEXT)
room_types: id (INTEGER), hotel_id (INTEGER), room_code (TEXT), name (TEXT), max_guests (INTEGER), description (TEXT), is_active (INTEGER)
rooms: id (INTEGER), room_type_id (IN

In [3]:
%run code/staff_main.py

In [4]:
%run code/platform_main.py

In [5]:
from pathlib import Path

task_root = Path.cwd()
code_folder = task_root / "code"

if not code_folder.exists():
    raise FileNotFoundError(
        f"当前目录下没有 code 文件夹，请确认打开的是克隆目录中的 task3：{task_root}"
    )

ignore_path = task_root / ".gitignore"

ignore_entries = [
    "data/*.db",
    "data/*.sqlite",
    "data/*.sqlite3",
    "__pycache__/",
    "code/__pycache__/",
    ".ipynb_checkpoints/",
]

existing_entries = []
if ignore_path.exists():
    existing_entries = ignore_path.read_text(encoding="utf-8").splitlines()

for entry in ignore_entries:
    if entry not in existing_entries:
        existing_entries.append(entry)

ignore_path.write_text(
    "\n".join(existing_entries) + "\n",
    encoding="utf-8",
)

print("已创建或更新 .gitignore：", ignore_path)
print()

python_files = sorted(code_folder.rglob("*.py"))
total_lines = 0
non_comment_lines = 0

for file_path in python_files:
    lines = file_path.read_text(encoding="utf-8").splitlines()
    file_lines = len(lines)
    file_code_lines = sum(
        1
        for line in lines
        if line.strip() and not line.lstrip().startswith("#")
    )

    total_lines += file_lines
    non_comment_lines += file_code_lines

    print(
        f"{file_path.name}: "
        f"总行数 {file_lines}，非空且非注释行 {file_code_lines}"
    )

print()
print("Python 文件总数：", len(python_files))
print("总行数：", total_lines)
print("非空且非注释行：", non_comment_lines)

已创建或更新 .gitignore： C:\Users\34974\Documents\GitHub\hotel-reservation-system\task3\.gitignore

auth_service-checkpoint.py: 总行数 132，非空且非注释行 99
booking_service-checkpoint.py: 总行数 337，非空且非注释行 279
database-checkpoint.py: 总行数 143，非空且非注释行 121
hotel_service-checkpoint.py: 总行数 204，非空且非注释行 170
main-checkpoint.py: 总行数 18，非空且非注释行 9
models-checkpoint.py: 总行数 84，非空且非注释行 65
platform_main-checkpoint.py: 总行数 21，非空且非注释行 13
platform_service-checkpoint.py: 总行数 777，非空且非注释行 638
platform_ui-checkpoint.py: 总行数 902，非空且非注释行 775
staff_main-checkpoint.py: 总行数 22，非空且非注释行 13
staff_service-checkpoint.py: 总行数 790，非空且非注释行 640
staff_ui-checkpoint.py: 总行数 914，非空且非注释行 784
ui-checkpoint.py: 总行数 1038，非空且非注释行 893
auth_service.py: 总行数 132，非空且非注释行 99
booking_service.py: 总行数 337，非空且非注释行 279
database.py: 总行数 143，非空且非注释行 121
hotel_service.py: 总行数 204，非空且非注释行 170
main.py: 总行数 18，非空且非注释行 9
models.py: 总行数 84，非空且非注释行 65
platform_main.py: 总行数 21，非空且非注释行 13
platform_service.py: 总行数 777，非空且非注释行 638
platform_ui.py: 总行数 902，非空且非注释行 775
s